In [ ]:
    ############    #############   AsyncIO and async-await   #############   ##############   

 =>  async def defines a coroutine function; calling it returns a coroutine object, it does
       not run the body immediately.

 =>  await suspends the current coroutine and hands control back to the event loop until the
       awaited thing (another coroutine, a socket read, a sleep) is ready.

 =>  The event loop is a single-threaded scheduler. It runs whichever coroutine is ready and
       switches away the moment one awaits something that isn't ready yet.

 =>  asyncio.gather(*coros) runs coroutines concurrently and waits for all of them.

      Syntax -->

                async def fetch(name, delay):
                      await asyncio.sleep(delay)
                      return name

                await asyncio.gather(fetch('a', 1), fetch('b', 1))


In [ ]:
import asyncio
import time

async def fetch(name: str, delay: float) -> str:
    print(f"[{name}] starting, will take {delay}s")
    await asyncio.sleep(delay)
    print(f"[{name}] done")
    return name

async def main():
    start = time.perf_counter()
    results = await asyncio.gather(fetch("a", 1), fetch("b", 1), fetch("c", 1))
    elapsed = time.perf_counter() - start
    print("results:", results)
    print(f"elapsed: {elapsed:.2f}s (would be ~3s if run sequentially)")

await main()  # in a .py script use: asyncio.run(main())


<img src="images/asyncio-event-loop.png" alt="Sequential vs concurrent execution timeline">

In [ ]:
 =>  All three fetch() calls start almost immediately, then sleep concurrently, so total
       elapsed time is ~1s instead of ~3s.

 =>  In a real service, 'await asyncio.sleep()' is standing in for any I/O wait: a DB query,
       an HTTP call to an LLM provider, a file read.

 =>  Rule of thumb: async helps when you are waiting on I/O, not when you are burning CPU.
       CPU-bound work still blocks the single event-loop thread unless offloaded (see the
       Concurrency vs parallelism notebook).


In [ ]:
    ############    #############   Cancellation   #############   ##############   

 =>  Any running task can be cancelled with task.cancel() -- this raises
       asyncio.CancelledError inside the coroutine at its next suspension point (the next
       'await'), not instantly.

 =>  A coroutine SHOULD let CancelledError propagate (or re-raise after cleanup) rather than
       swallowing it -- catching it silently makes the task look 'done' when it was actually
       cancelled, which hides bugs.


In [ ]:
import asyncio

async def long_running_job():
    try:
        print("job: starting a long operation...")
        await asyncio.sleep(10)   # would take 10s if allowed to finish
        print("job: finished (you should not see this)")
    except asyncio.CancelledError:
        print("job: cleanup on cancellation (closing a connection, etc.)")
        raise   # re-raise so the caller/task correctly sees this as cancelled, not completed

async def main():
    task = asyncio.create_task(long_running_job())
    await asyncio.sleep(0.2)      # let it start
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        print("main: confirmed the task was cancelled")

await main()


In [ ]:
 =>  Notice the job only gets as far as its cleanup branch -- the 10s sleep never
       completes, and re-raising CancelledError means 'await task' in main() also sees it
       as cancelled (not as a normal return value).

 =>  This is the general-purpose version of the pattern; the 'Async Client Disconnection and
       Inference Cancellation' notebook later in this topic applies the exact same idea to a
       real production case: cancelling an upstream LLM call when an HTTP client disconnects
       mid-stream.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Replace asyncio.sleep with a real async HTTP call (httpx.AsyncClient) to a public
           API and time 3 concurrent calls vs 3 sequential ones.

 =>  [ ] Build a tiny FastAPI endpoint that awaits 2 slow calls with asyncio.gather instead
           of one after another.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Calling a blocking function (requests.get, time.sleep, a sync DB driver) inside an
       async def -- this blocks the ENTIRE event loop, freezing every other coroutine too.

 =>  Forgetting to await a coroutine -- Python just creates a coroutine object and warns
       'coroutine was never awaited', the code inside never runs.

 =>  Assuming asyncio.gather magically uses multiple CPU cores -- it doesn't; it's
       concurrency on one thread, not parallelism (next notebook).
